### Imports

In [ ]:
# BLS Data Import
from datetime import datetime
import requests
import json
import pandas as pd

# .env management
from dotenv import load_dotenv
import os

# Plots
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

### .env Management

In [ ]:
load_dotenv("../.env")
api_key = os.getenv("BLS_KEY")

### Job Data Import

Job Openings and Labor Turnover Survey
https://data.bls.gov/timeseries/JTS000000000000000HIR

Series Id:	JTS000000000000000HIR
Seasonally adjusted
Industry:	Total nonfarm
State/Region:	Total US
Area:	All areas
Data Element:	Hires
Size Class:	All size classes
Rate/Level:	Rate

In [ ]:
rows = []
# Account for 20 year max pull from BLS
def split_range(start, end, max_length=20):
    segments = []
    while start <= end:
        segment_end = min(start + max_length - 1, end)
        segments.append((start, segment_end))
        start = segment_end + 1
    return segments
segments = split_range(2000, datetime.now().year, 20)

for startyear, endyear in segments:
    headers = {'Content-type': 'application/json'}
    data = json.dumps({"seriesid": ['JTS000000000000000HIR'],
                    "startyear": str(startyear), 
                    "endyear": str(endyear),
                    "registrationkey": api_key})
    p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', 
                      data = data, 
                      headers = headers)
    json_data = json.loads(p.text)

    for series in json_data['Results']['series']:
        # Monitor warnings ("Unable to get Catalog" does not affect data quality)
        print(json_data['message'])
        seriesID = series['seriesID']
        for item in series['data']:
            year = item['year']
            period = item['period']
            month_name = item['periodName']
            value = item['value']
            footnotes = ", ".join(f['text'] for f in item['footnotes'] if f)
            # Only save row if monthly entry
            if 'M01' <= period <= 'M12':
                rows.append({
                    "year": year,
                    "period": period,
                    "month_name": month_name,
                    "value": value,
                    "footnotes": footnotes
                })
df = pd.DataFrame(rows)
df = df.sort_values(["year", "period"])

['Unable to get Catalog Data for series JTS000000000000000HIR']
['Unable to get Catalog Data for series JTS000000000000000HIR']


In [ ]:
df.to_csv("../../../data/bls/BLS_JobOpenings.csv", index = False)

In [ ]:
df = pd.read_csv("../../../data/bls/BLS_JobOpenings.csv")
df

,year,period,month_name,value,footnotes
0,2000,M12,December,4.1,NaN
1,2001,M01,January,4.3,NaN
2,2001,M02,February,4.0,NaN
3,2001,M03,March,4.2,NaN
4,2001,M04,April,3.9,NaN
...,...,...,...,...,...
298,2025,M10,October,3.3,NaN
299,2025,M11,November,3.2,NaN
300,2025,M12,December,3.3,NaN
301,2026,M01,January,3.4,NaN


### Plots